# Домашнее задание 6

Данные о клиентах с веб-портала — `web_clients_correct.csv`, беру первые 15 строк.

Две задачи: заменить числовой возраст текстовыми категориями через `.apply` и отобрать
клиентов по двум условиям сразу.

## Подготовка

Файл лежит в этой же папке. Импортирую pandas и расширяю вывод — иначе таблица не влезает
в экран и часть столбцов обрезается.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [2]:
from pathlib import Path


def check_input(file_path, columns=("sex", "age")):
    """Проверяю, что файл существует и нужные столбцы в нём есть."""
    if not Path(file_path).exists():
        raise FileNotFoundError(f"Нет файла {file_path}: положите его рядом с ноутбуком.")
    return file_path


def load_clients(file_path, rows=15):
    """Загружаю первые rows строк из CSV."""
    check_input(file_path)
    return pd.read_csv(file_path, nrows=rows, encoding="utf-8")


def classify_age(age):
    """Возвращаю категорию для одного значения возраста."""
    if age > 25:
        return "Старше 25"
    return "Младше или равен 25"


def mark_age(clients, column="age"):
    """Применяю classify_age к столбцу через .apply и меняю данные в копии."""
    result = clients.copy()
    result[column] = result[column].apply(classify_age)
    return result


def filter_clients(clients, sex="male", min_age=30):
    """Отбираю строки по полу и возрасту (возраст строго больше min_age)."""
    mask = (clients["sex"] == sex) & (clients["age"] > min_age)
    return clients[mask].copy()


## Задание 1. Категории возраста

Загружаю 15 строк и заменяю каждый возраст текстом: больше 25 лет — «Старше 25», иначе —
«Младше или равен 25». Сравнение строгое, поэтому клиент ровно 25 лет попадает во вторую
категорию.


In [3]:
def task_1(file_path):
    clients = load_clients(file_path)
    categorized = mark_age(clients)
    print(categorized["age"].to_string())


task_1("web_clients_correct.csv")


0               Старше 25
1               Старше 25
2               Старше 25
3               Старше 25
4     Младше или равен 25
5               Старше 25
6               Старше 25
7               Старше 25
8               Старше 25
9               Старше 25
10              Старше 25
11              Старше 25
12    Младше или равен 25
13              Старше 25
14              Старше 25


## Задание 2. Мужчины старше 30 лет

Отбираю строки по двум условиям одновременно: пол `male` и возраст строго больше 30.
Объединяю их через `&`, каждое беру в скобки — иначе приоритет операторов сработает
не так, как нужно. Клиент, которому ровно 30, в выборку не попадёт.


In [4]:
def task_2(file_path):
    clients = load_clients(file_path)
    selected = filter_clients(clients)
    print(selected.to_string())
    print(f"\nОтобрано строк: {len(selected)}")


task_2("web_clients_correct.csv")


                                    name device_type            browser   sex  age  bill                           region
1         Allison  Master. Hudson Trevor      tablet              Opera  male   48   850  Montreal: PQ / Chesterville: ON
5                    Anderson  Mr. Harry      mobile            Firefox  male   48   371                     New York: NY
7                 Andrews  Mr. Thomas Jr      tablet  Internet Explorer  male   39   867                      Belfast: NI
9                Artagaveytia  Mr. Ramon      mobile            Firefox  male   71   926              Montevideo: Uruguay
10                Astor  Col. John Jacob     desktop             Chrome  male   47   100                     New York: NY
14  Barkworth  Mr. Algernon Henry Wilson      laptop             Chrome  male   80   723                    Hessle: Yorks

Отобрано строк: 6


Обе границы в задании строгие. Будь сравнение нестрогим, клиентка 25 лет ушла бы
в первую категорию, а клиент 30 лет попал бы в отбор — поэтому везде стоит `>`.

## Проверка

Прогоняю проверки на границах 25 и 26 лет, сверяю индексы и возрасты отобранных строк
и убеждаюсь, что исходная таблица не изменилась.

In [5]:
def self_check(file_path):
    clients = load_clients(file_path)
    original = clients.copy(deep=True)

    expected = [
        "Старше 25", "Старше 25", "Старше 25", "Старше 25",
        "Младше или равен 25", "Старше 25", "Старше 25",
        "Старше 25", "Старше 25", "Старше 25", "Старше 25",
        "Старше 25", "Младше или равен 25", "Старше 25", "Старше 25",
    ]
    assert len(clients) == 15
    assert mark_age(clients)["age"].tolist() == expected
    assert classify_age(25) == "Младше или равен 25"
    assert classify_age(26) == "Старше 25"

    selected = filter_clients(clients)
    assert selected.index.tolist() == [1, 5, 7, 9, 10, 14]
    assert selected["age"].tolist() == [48, 48, 39, 71, 47, 80]
    assert (selected["sex"] == "male").all()
    assert selected.columns.tolist() == clients.columns.tolist()

    boundary = pd.DataFrame({"sex": ["male", "male", "female"], "age": [30, 31, 31]})
    assert filter_clients(boundary).index.tolist() == [1]
    pd.testing.assert_frame_equal(clients, original)

    print("Все проверки пройдены.")


self_check("web_clients_correct.csv")


Все проверки пройдены.
